<a href="https://colab.research.google.com/github/Castlebin/d2l-zh-pytorch-colab/blob/my_master/9_d2l-zh-pytorch-colab-reorg/05_%E6%B7%B1%E5%BA%A6%E5%AD%A6%E4%B9%A0%E8%AE%A1%E7%AE%97/01_%E6%A8%A1%E5%9E%8B%E6%9E%84%E5%BB%BA%E4%B8%8E%E5%8F%82%E6%95%B0%E7%AE%A1%E7%90%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 模型构建与参数管理

本notebook介绍PyTorch中模型构建和参数管理的核心技术:
- 层和块(Module)的概念
- 自定义层和模型
- 参数访问和初始化
- 参数共享
- 模型的保存和加载

掌握这些技术,你就能构建任意复杂的神经网络!

## 第一部分: 层和块

### 1.1 什么是块(Block)?

**块**是PyTorch中的核心抽象概念:
- **单个层**: 如`nn.Linear`, `nn.Conv2d`
- **多个层的组合**: 如ResNet中的残差块
- **整个模型**: 完整的神经网络

**块的特点**:
1. 接受输入数据
2. 生成输出
3. 包含可学习的参数
4. 可以计算梯度(自动微分)

**块的组合**:
```
简单块 → 组合成复杂块 → 组合成更复杂的块 → 完整模型
```

### 1.2 使用Sequential构建模型

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
# Sequential: 按顺序执行的容器
net = nn.Sequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

X = torch.rand(2, 20)
output = net(X)

print(f'输入形状: {X.shape}')
print(f'输出形状: {output.shape}')
print(f'\n模型结构:')
print(net)

输入形状: torch.Size([2, 20])
输出形状: torch.Size([2, 10])

模型结构:
Sequential(
  (0): Linear(in_features=20, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)


**Sequential的工作原理**:
- 维护一个有序的Module列表
- 按顺序执行每个模块
- 每个模块的输出是下一个模块的输入
- `net(X)` 实际上调用 `net.__call__(X)`

### 1.3 自定义块

**为什么需要自定义块?**
- Sequential只能按顺序执行
- 有些模型需要更复杂的控制流
- 需要实现特殊的计算逻辑

**自定义块的要求**:
1. 继承`nn.Module`
2. 在`__init__`中定义层和参数
3. 实现`forward`方法定义前向传播

In [3]:
# 自定义MLP块
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        # 自定义前向传播逻辑
        return self.out(F.relu(self.hidden(X)))

net_custom = MLP()
print('自定义MLP:')
print(net_custom)
print(f'\n输出形状: {net_custom(X).shape}')

自定义MLP:
MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (out): Linear(in_features=256, out_features=10, bias=True)
)

输出形状: torch.Size([2, 10])


### 1.4 在forward中使用控制流

In [4]:
class FlexibleMLP(nn.Module):
    """带有控制流的MLP"""
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(20, 256)
        self.linear2 = nn.Linear(256, 256)
        self.linear3 = nn.Linear(256, 10)

    def forward(self, X):
        H = F.relu(self.linear1(X))

        # 控制流: 条件执行
        if H.abs().sum() > 1:
            H = F.relu(self.linear2(H))  # 额外的层
        else:
            H = H / 2  # 缩放

        return self.linear3(H)

flexible_net = FlexibleMLP()
print('灵活的MLP(带控制流):')
print(flexible_net(X).shape)

灵活的MLP(带控制流):
torch.Size([2, 10])


### 1.5 嵌套块

In [5]:
def make_block(in_features, out_features):
    # Helper function to create a block with specified input/output features
    return nn.Sequential(
        nn.Linear(in_features, 64),
        nn.ReLU(),
        nn.Linear(64, out_features),
        nn.ReLU()
    )

def block2():
    net = nn.Sequential()
    # The first block receives 20 features from the input and outputs 32
    net.add_module('block0', make_block(20, 32))
    # Subsequent blocks receive 32 features (from the previous block) and output 32
    for i in range(1, 4):
        net.add_module(f'block{i}', make_block(32, 32))
    return net

# 构建嵌套网络
nested_net = nn.Sequential(block2(), nn.Linear(32, 10))

print('嵌套网络结构:')
print(nested_net)
print(f'\n输出形状: {nested_net(torch.rand(2, 20)).shape}')

嵌套网络结构:
Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=20, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=32, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=32, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=32, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=32, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=32, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=32, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=32, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=32, out_features=10, bias=True)
)

输出形状: torch.Size([2, 10])


---

## 第二部分: 参数管理

### 2.1 参数访问

**为什么需要访问参数?**
- 调试和诊断
- 可视化权重
- 迁移学习
- 参数剪枝

In [6]:
# 创建一个简单的网络
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

X = torch.rand(2, 4)
print('前向传播输出:')
print(net(X))

前向传播输出:
tensor([[-0.4617],
        [-0.4365]], grad_fn=<AddmmBackward0>)


**访问特定层的参数**

In [7]:
# 访问第3个模块(第二个线性层)的参数
print('第二个线性层的参数:')
print(net[2].state_dict())
print(f'\n权重形状: {net[2].weight.shape}')
print(f'偏置形状: {net[2].bias.shape}')

第二个线性层的参数:
OrderedDict({'weight': tensor([[ 0.0257, -0.0282,  0.1402, -0.3514, -0.3225,  0.1154,  0.2132,  0.3034]]), 'bias': tensor([-0.0765])})

权重形状: torch.Size([1, 8])
偏置形状: torch.Size([1])


**访问参数的值和梯度**

In [8]:
# 参数类型
print(f'参数类型: {type(net[2].bias)}')

# 参数值
print(f'\n偏置参数: {net[2].bias}')
print(f'偏置值: {net[2].bias.data}')

# 梯度(训练前为None)
print(f'\n梯度是否为None: {net[2].weight.grad is None}')

参数类型: <class 'torch.nn.parameter.Parameter'>

偏置参数: Parameter containing:
tensor([-0.0765], requires_grad=True)
偏置值: tensor([-0.0765])

梯度是否为None: True


In [9]:
# 进行一次反向传播后查看梯度
net(X).sum().backward()
print('反向传播后的梯度:')
print(net[2].weight.grad)

反向传播后的梯度:
tensor([[0.0000, 1.2882, 0.3507, 1.5376, 0.6775, 0.0065, 0.0000, 0.0000]])


### 2.2 一次性访问所有参数

In [10]:
# 访问第一层的参数
print('第一层的参数:')
print(*[(name, param.shape) for name, param in net[0].named_parameters()])

# 访问所有层的参数
print('\n所有层的参数:')
print(*[(name, param.shape) for name, param in net.named_parameters()])

第一层的参数:
('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))

所有层的参数:
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [11]:
# 通过state_dict访问
print('\n通过state_dict访问:')
print(net.state_dict().keys())
print(f'\n第二层偏置: {net.state_dict()["2.bias"].data}')


通过state_dict访问:
odict_keys(['0.weight', '0.bias', '2.weight', '2.bias'])

第二层偏置: tensor([-0.0765])


### 2.3 嵌套块的参数访问

In [12]:
# 访问嵌套网络的参数
print('嵌套网络的所有参数名称:')
for name, param in nested_net.named_parameters():
    print(f'{name}: {param.shape}')

嵌套网络的所有参数名称:
0.block0.0.weight: torch.Size([64, 20])
0.block0.0.bias: torch.Size([64])
0.block0.2.weight: torch.Size([32, 64])
0.block0.2.bias: torch.Size([32])
0.block1.0.weight: torch.Size([64, 32])
0.block1.0.bias: torch.Size([64])
0.block1.2.weight: torch.Size([32, 64])
0.block1.2.bias: torch.Size([32])
0.block2.0.weight: torch.Size([64, 32])
0.block2.0.bias: torch.Size([64])
0.block2.2.weight: torch.Size([32, 64])
0.block2.2.bias: torch.Size([32])
0.block3.0.weight: torch.Size([64, 32])
0.block3.0.bias: torch.Size([64])
0.block3.2.weight: torch.Size([32, 64])
0.block3.2.bias: torch.Size([32])
1.weight: torch.Size([10, 32])
1.bias: torch.Size([10])


---

## 第三部分: 参数初始化

### 3.1 内置初始化方法

In [13]:
# 创建一个新网络
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

# 1. 正态分布初始化
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)

net.apply(init_normal)
print('正态分布初始化后的权重:')
print(net[0].weight.data[0])

正态分布初始化后的权重:
tensor([-0.0064,  0.0110, -0.0018,  0.0027])


In [14]:
# 2. 常数初始化
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
print('常数初始化后的权重:')
print(net[0].weight.data[0])

常数初始化后的权重:
tensor([1., 1., 1., 1.])


In [15]:
# 3. Xavier初始化(适合Sigmoid/Tanh)
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

net.apply(init_xavier)
print('Xavier初始化后的权重:')
print(net[0].weight.data[0])

Xavier初始化后的权重:
tensor([-0.5471, -0.1371, -0.5539,  0.3747])


In [16]:
# 4. He初始化(适合ReLU)
def init_he(m):
    if type(m) == nn.Linear:
        nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')

net.apply(init_he)
print('He初始化后的权重:')
print(net[0].weight.data[0])

He初始化后的权重:
tensor([ 0.0967,  0.3775,  0.1846, -0.0643])


### 3.2 对不同层使用不同的初始化

In [17]:
def init_custom(m):
    if type(m) == nn.Linear:
        print(f'初始化 {m}')
        nn.init.xavier_uniform_(m.weight)

# 只对第一层应用Xavier
net[0].apply(init_xavier)
# 对第三层应用He初始化
net[2].apply(init_he)

print('\n第一层权重:', net[0].weight.data[0][:3])
print('第二层权重:', net[2].weight.data[0][:3])


第一层权重: tensor([ 0.0102,  0.1118, -0.6562])
第二层权重: tensor([ 0.0093, -0.5323, -0.3103])


### 3.3 自定义初始化

In [18]:
# 自定义初始化: 使用特殊的分布
def custom_init(m):
    if type(m) == nn.Linear:
        print(f'初始化 {m}')
        # 均匀分布 U(-10, 10), 但保留绝对值大于5的权重
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(custom_init)
print('\n自定义初始化后的权重:')
print(net[0].weight[:2])

初始化 Linear(in_features=4, out_features=8, bias=True)
初始化 Linear(in_features=8, out_features=1, bias=True)

自定义初始化后的权重:
tensor([[7.0982, 0.0000, 0.0000, -0.0000],
        [-0.0000, 0.0000, -0.0000, 7.1154]], grad_fn=<SliceBackward0>)


### 3.4 直接设置参数

In [19]:
# 直接修改参数
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42

print('直接设置后的权重:')
print(net[0].weight.data[0])

直接设置后的权重:
tensor([42.,  1.,  1.,  1.])


---

## 第四部分: 参数共享

**为什么需要参数共享?**
- 减少参数数量
- 提高模型效率
- 某些架构需要(如Siamese网络)

In [20]:
# 创建一个共享层
shared_layer = nn.Linear(8, 8)

# 使用共享层多次
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    shared_layer,  # 第一次使用
    nn.ReLU(),
    shared_layer,  # 第二次使用(共享参数)
    nn.ReLU(),
    nn.Linear(8, 1)
)

print('网络结构(注意layer 2和4是同一个对象):')
print(net)
print(f'\n总参数数量: {sum(p.numel() for p in net.parameters())}')

网络结构(注意layer 2和4是同一个对象):
Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=8, bias=True)
  (5): ReLU()
  (6): Linear(in_features=8, out_features=1, bias=True)
)

总参数数量: 121


In [21]:
# 验证参数是否共享
print('layer[2]和layer[4]是否是同一个对象?', net[2] is net[4])
print('\nlayer[2]的权重:')
print(net[2].weight.data[0][:5])
print('\nlayer[4]的权重(应该相同):')
print(net[4].weight.data[0][:5])

layer[2]和layer[4]是否是同一个对象? True

layer[2]的权重:
tensor([-0.2227,  0.2103,  0.1200,  0.0959, -0.1079])

layer[4]的权重(应该相同):
tensor([-0.2227,  0.2103,  0.1200,  0.0959, -0.1079])


In [22]:
# 修改共享层的参数
net[2].weight.data[0, 0] = 100

print('修改layer[2]后:')
print('layer[2]的权重:', net[2].weight.data[0, 0])
print('layer[4]的权重(也改变了):', net[4].weight.data[0, 0])

修改layer[2]后:
layer[2]的权重: tensor(100.)
layer[4]的权重(也改变了): tensor(100.)


---

## 第五部分: 自定义层

### 5.1 不带参数的自定义层

In [23]:
class CenteredLayer(nn.Module):
    """将输入减去均值"""
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

# 测试
layer = CenteredLayer()
test_input = torch.FloatTensor([1, 2, 3, 4, 5])
print('输入:', test_input)
print('输出(减去均值):', layer(test_input))
print('输出的均值(应该接近0):', layer(test_input).mean())

输入: tensor([1., 2., 3., 4., 5.])
输出(减去均值): tensor([-2., -1.,  0.,  1.,  2.])
输出的均值(应该接近0): tensor(0.)


In [24]:
# 将自定义层嵌入到网络中
net = nn.Sequential(
    nn.Linear(8, 128),
    CenteredLayer()
)

Y = net(torch.rand(4, 8))
print(f'输出的均值: {Y.mean():.6f}')

输出的均值: 0.000000


### 5.2 带参数的自定义层

In [25]:
class MyLinear(nn.Module):
    """自定义全连接层"""
    def __init__(self, in_units, units):
        super().__init__()
        # 使用nn.Parameter定义参数
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

# 测试
linear = MyLinear(5, 3)
print('自定义线性层:')
print(f'权重形状: {linear.weight.shape}')
print(f'偏置形状: {linear.bias.shape}')

# 前向传播
test_input = torch.rand(2, 5)
output = linear(test_input)
print(f'\n输入形状: {test_input.shape}')
print(f'输出形状: {output.shape}')

自定义线性层:
权重形状: torch.Size([5, 3])
偏置形状: torch.Size([3])

输入形状: torch.Size([2, 5])
输出形状: torch.Size([2, 3])


In [26]:
# 使用自定义层构建网络
net = nn.Sequential(
    MyLinear(64, 8),
    MyLinear(8, 1)
)

print('使用自定义层的网络:')
print(net)
print(f'\n输出形状: {net(torch.rand(2, 64)).shape}')

使用自定义层的网络:
Sequential(
  (0): MyLinear()
  (1): MyLinear()
)

输出形状: torch.Size([2, 1])


---

## 第六部分: 保存和加载模型

### 6.1 保存和加载张量

In [27]:
import os

# 创建保存目录
os.makedirs('../data', exist_ok=True)

# 保存单个张量
x = torch.arange(4)
torch.save(x, '../data/x-tensor.pt')

# 加载
x2 = torch.load('../data/x-tensor.pt')
print('保存并加载的张量:')
print(x2)

保存并加载的张量:
tensor([0, 1, 2, 3])


In [28]:
# 保存张量列表
y = torch.zeros(4)
torch.save([x, y], '../data/xy-tensors.pt')

x2, y2 = torch.load('../data/xy-tensors.pt')
print('加载的张量列表:')
print('x:', x2)
print('y:', y2)

加载的张量列表:
x: tensor([0, 1, 2, 3])
y: tensor([0., 0., 0., 0.])


In [29]:
# 保存字典
mydict = {'x': x, 'y': y}
torch.save(mydict, '../data/mydict.pt')

mydict2 = torch.load('../data/mydict.pt')
print('加载的字典:')
print(mydict2)

加载的字典:
{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}


### 6.2 保存和加载模型参数

In [30]:
# 定义一个MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(2, 20)
Y = net(X)

print('原始模型输出:')
print(Y)

原始模型输出:
tensor([[ 0.2008, -0.0644, -0.2507, -0.2717, -0.2480,  0.2171, -0.0931, -0.1129,
          0.2539,  0.0698],
        [-0.2614,  0.0310,  0.0166, -0.1107, -0.0856,  0.2597, -0.3479,  0.2217,
          0.3394,  0.0082]], grad_fn=<AddmmBackward0>)


In [31]:
# 保存模型参数
torch.save(net.state_dict(), '../data/mlp.params')
print('模型参数已保存到 ../data/mlp.params')

模型参数已保存到 ../data/mlp.params


In [32]:
# 加载模型参数
clone = MLP()
clone.load_state_dict(torch.load('../data/mlp.params'))
clone.eval()  # 设置为评估模式

Y_clone = clone(X)
print('克隆模型输出(应该与原始相同):')
print(Y_clone)
print(f'\n输出是否相同: {(Y == Y_clone).all()}')

克隆模型输出(应该与原始相同):
tensor([[ 0.2008, -0.0644, -0.2507, -0.2717, -0.2480,  0.2171, -0.0931, -0.1129,
          0.2539,  0.0698],
        [-0.2614,  0.0310,  0.0166, -0.1107, -0.0856,  0.2597, -0.3479,  0.2217,
          0.3394,  0.0082]], grad_fn=<AddmmBackward0>)

输出是否相同: True


**注意**:
- `torch.save(net.state_dict())` 只保存参数,不保存模型结构
- 加载时需要先定义相同的模型结构
- 这是推荐的做法,因为模型代码可能包含任意Python代码

### 6.3 保存整个模型(不推荐)

In [33]:
# 保存整个模型(包括结构)
torch.save(net, '../data/mlp-entire.pt')

# 加载
# 为了解决 PyTorch 2.6+ 的 UnpicklingError，需要设置 weights_only=False
# 注意：这允许加载任意 Python 代码，请确保模型来源可信。
loaded_net = torch.load('../data/mlp-entire.pt', weights_only=False)
print('加载的完整模型:')
print(loaded_net)
print(f'\n输出: {loaded_net(X)}')


加载的完整模型:
MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

输出: tensor([[ 0.2008, -0.0644, -0.2507, -0.2717, -0.2480,  0.2171, -0.0931, -0.1129,
          0.2539,  0.0698],
        [-0.2614,  0.0310,  0.0166, -0.1107, -0.0856,  0.2597, -0.3479,  0.2217,
          0.3394,  0.0082]], grad_fn=<AddmmBackward0>)


---

## 小结

### 核心概念

1. **块(Module)**:
   - PyTorch的核心抽象
   - 可以是单层、多层组合或完整模型
   - 必须实现`__init__`和`forward`

2. **模型构建**:
   - `nn.Sequential`: 顺序执行
   - 自定义Module: 灵活的控制流
   - 嵌套块: 构建复杂架构

3. **参数管理**:
   - `state_dict()`: 获取所有参数
   - `named_parameters()`: 遍历参数
   - `apply()`: 批量应用函数

4. **参数初始化**:
   - Xavier: 适合Sigmoid/Tanh
   - He(Kaiming): 适合ReLU
   - 自定义初始化: 灵活控制

5. **参数共享**:
   - 使用同一个层对象
   - 减少参数量
   - 梯度会累积

6. **自定义层**:
   - 继承`nn.Module`
   - 使用`nn.Parameter`定义可学习参数
   - 实现`forward`方法

7. **模型保存**:
   - 推荐: `torch.save(model.state_dict())`
   - 加载: `model.load_state_dict(torch.load())`

### 最佳实践

```python
# 1. 定义模型
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 20)
        self.layer2 = nn.Linear(20, 10)
    
    def forward(self, x):
        return self.layer2(F.relu(self.layer1(x)))

# 2. 初始化
model = MyModel()
model.apply(lambda m: nn.init.xavier_uniform_(m.weight) if isinstance(m, nn.Linear) else None)

# 3. 训练
# ...

# 4. 保存
torch.save(model.state_dict(), 'model.pt')

# 5. 加载
model = MyModel()
model.load_state_dict(torch.load('model.pt'))
model.eval()
```

## 练习

1. **构建ResNet块**: 实现一个残差块(带跳跃连接)
2. **参数统计**: 编写函数统计模型的总参数量
3. **参数冻结**: 实现冻结某些层的参数(迁移学习)
4. **权重可视化**: 可视化第一层卷积核的权重
5. **自定义激活函数**: 实现Swish激活函数
6. **检查点**: 实现训练过程中定期保存模型
7. **模型比较**: 比较两个模型的参数是否相同
8. **参数裁剪**: 实现权重裁剪(将小于阈值的权重置零)